# Estudo do público

## Bibliotecas e leitura da base

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when

In [ ]:
df_cadastro = spark.read.parquet(
    '/Volumes/hackathon_2025/default/source/base_dados_cadastrais/'
)
df_cadastro.createOrReplaceTempView("df_cadastro")
display(df_cadastro)

## Quantidade de clientes

In [ ]:
# Executar a query e converter para pandas
qtd_cliente = spark.sql("""
select safra,
      count(distinct NUM_CPF) as qtd_cliente
from df_cadastro
where FPD is not null and flag_mig2 = "PRE" and FLAG_INSTALACAO = 1
group by safra
order by safra
""")

qtd_cliente = qtd_cliente.toPandas()

# Renomear os valores da coluna safra
mapeamento = {
    '202410': 'nov-2024',
    '202411': 'out-2024',
    '202412': 'dez-2024',
    '202501': 'jan-2025',
    '202502': 'fev-2025',
    '202503': 'mar-2025'
}

qtd_cliente['safra'] = qtd_cliente['safra'].replace(mapeamento)
display(qtd_cliente)

### Visualização

In [ ]:
# Criar gráfico de barras verticais
plt.figure(figsize=(10, 6))
ax = sns.barplot(data=qtd_cliente, x='safra', y='qtd_cliente', color='#6fa8dc')
plt.xlabel('Mês')
plt.ylabel('Quantidade de Clientes')
plt.title('Quantidade de Clientes por mês')
plt.grid(True, alpha=0.3, axis='y')

# Adicionar valores em cima das barras
for p in ax.patches:
    height = p.get_height()
    ax.text(p.get_x() + p.get_width()/2., height,
            f'{int(height)}',
            ha="center", va="bottom", fontsize=11)

plt.tight_layout()
display(plt.show())

## Quantidade de inadimplentes vs adimplentes

In [ ]:
# Executar a query e converter para pandas
qtd_cliente = spark.sql("""
select safra,
      count(distinct case when FPD = 1 then NUM_CPF end) as qtd_inadimplentes,
      count(distinct case when FPD = 0 then NUM_CPF end) as qtd_adimplentes,
      count(distinct NUM_CPF) as qtd_total,
      round(count(distinct case when FPD = 1 then NUM_CPF end) / count(distinct NUM_CPF), 4) as taxa_inadimplentes,
      round(count(distinct case when FPD = 0 then NUM_CPF end) / count(distinct NUM_CPF), 4) as taxa_adimplentes
from df_cadastro
where FPD is not null and flag_mig2 = "PRE" and FLAG_INSTALACAO = 1
group by safra
order by safra
""")
display(qtd_cliente)
qtd_cliente = qtd_cliente.toPandas()

## Idade

In [ ]:
%sql
select idade,
      count(*) as qtd
from df_cadastro
where FPD is not null and flag_mig2 = "PRE"
group by idade
order by idade

In [ ]:
# Aplicar os mesmos filtros
idade_filtrada = df_cadastro.filter(
    (F.col("FPD").isNotNull()) &
    (F.col("flag_mig2") == "PRE")
).select("idade").toPandas()

# Remover nulos
idade_limpa = idade_filtrada['idade'].dropna()

# Fazer describe
print(idade_limpa.describe())

### Visualização

In [ ]:
# Criar histograma para a variável idade

# Aplicar os mesmos filtros da query SQL
idade_filtrada = df_cadastro.filter(
    (F.col("FPD").isNotNull()) &
    (F.col("flag_mig2") == "PRE")
).select("idade").toPandas()

# Remover nulos
idade_limpa = idade_filtrada['idade'].dropna()

# Criar histograma
plt.figure(figsize=(12, 6))
plt.hist(idade_limpa, bins=50, edgecolor='black', color='#6fa8dc', alpha=0.8)

plt.xlabel('Idade', fontsize=26)
plt.ylabel('Frequência', fontsize=26)
plt.title('Distribuição de Idade', fontsize=28)
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)

# Remover bordas
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Criar histograma para idade menor que 18
idade_baixa = idade_limpa[idade_limpa < 18]
plt.hist(idade_baixa, bins=11, edgecolor='black', color='#6fa8dc')

plt.xlabel('Idade', fontsize=20)
plt.ylabel('Frequência', fontsize=20)
plt.title('Distribuição de Idade \nmenor do que 18', fontsize=20)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)

# Remover bordas
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

In [ ]:
# Criar histograma para idade maior que 90
idade_baixa = idade_limpa[idade_limpa > 90]
plt.hist(idade_baixa, bins=11, edgecolor='black', color='#6fa8dc')

plt.xlabel('Idade', fontsize=20)
plt.ylabel('Frequência', fontsize=20)
plt.title('Distribuição de Idade \nmaior do que 90', fontsize=20)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)

# Remover bordas
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

In [ ]:
# Criar boxplot para a variável idade

# Aplicar os mesmos filtros da query SQL
idade_filtrada = df_cadastro.filter(
    (F.col("FPD").isNotNull()) &
    (F.col("flag_mig2") == "PRE")
).select("idade").toPandas()

# Remover nulos
idade_limpa = idade_filtrada['idade'].dropna()

# Criar boxplot
plt.figure(figsize=(10, 6))
bp = plt.boxplot(idade_limpa, vert=True, patch_artist=True, widths=0.3)
bp['boxes'][0].set_facecolor('#6fa8dc')
bp['medians'][0].set_color('black')
bp['medians'][0].set_linewidth(2)

plt.ylabel('Idade', fontsize=20)
plt.title('Distribuição de Idade', fontsize=26)
plt.xticks([])
plt.yticks(fontsize=18)

# Remover bordas
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Criar gráfico scatter do volume de clientes por idade vs FPD

# Aplicar filtros
df_idade_fpd = df_cadastro.filter(
    (F.col("FPD").isNotNull()) &
    (F.col("flag_mig2") == "PRE")
).select("idade", "FPD").toPandas()

# Converter FPD para numérico
df_idade_fpd['FPD'] = pd.to_numeric(df_idade_fpd['FPD'], errors='coerce')

# Remover nulos
df_idade_fpd = df_idade_fpd.dropna()

# Agrupar por idade
fpd_idade = df_idade_fpd.groupby('idade').agg(
    volume=('FPD', 'count'),
    fpd_count=('FPD', 'sum')
).reset_index()

fpd_idade['taxa_fpd'] = fpd_idade['fpd_count'] / fpd_idade['volume']

# Filtrar volume mínimo (ajuste conforme necessário)
fpd_idade_filtro = fpd_idade[fpd_idade['volume'] >= 100]

plt.figure(figsize=(8, 5))
plt.scatter(fpd_idade_filtro['volume'], fpd_idade_filtro['taxa_fpd'], color='#b51f19')
plt.xlabel('Volume de Clientes', fontsize=14)
plt.ylabel('Taxa de FPD', fontsize=14)
plt.title('Volume de clientes por Idade vs Taxa de FPD', fontsize=16)

# Aumentar fonte dos ticks
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Remover bordas superior e direita
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## Localização regional

In [ ]:
# Tabela de referência de faixas de CEP
cep_ref = pd.DataFrame({
    "cep_inicio": [100,200,290,300,400,490,500,570,580,590,600,640,650,
                   660,689,690,693,694,699,700,737,768,770,780,789,790,
                   800,880,900],
    "cep_fim":   [199,289,299,399,489,499,569,579,589,599,639,649,659,
                   688,689,692,693,698,699,736,767,769,779,788,789,799,
                   879,899,999],
    "Estado": [
        "São Paulo","Rio de Janeiro","Espírito Santo","Minas Gerais",
        "Bahia","Sergipe","Pernambuco","Alagoas","Paraíba",
        "Rio Grande do Norte","Ceará","Piauí","Maranhão",
        "Pará","Amapá","Amazonas","Roraima","Amazonas","Acre",
        "Distrito Federal","Goiás","Rondônia","Tocantins",
        "Mato Grosso","Rondônia","Mato Grosso do Sul",
        "Paraná","Santa Catarina","Rio Grande do Sul"
    ],
    "Regiao": [
        "Sudeste","Sudeste","Sudeste","Sudeste",
        "Nordeste","Nordeste","Nordeste","Nordeste","Nordeste",
        "Nordeste","Nordeste","Nordeste","Nordeste",
        "Norte","Norte","Norte","Norte","Norte","Norte",
        "Centro-Oeste","Centro-Oeste","Norte","Norte",
        "Centro-Oeste","Norte","Centro-Oeste",
        "Sul","Sul","Sul"
    ]
})

def mapear_estado_regiao(cep3, tabela_ref):
    linha = tabela_ref[
        (tabela_ref["cep_inicio"] <= cep3) &
        (tabela_ref["cep_fim"] >= cep3)
    ]

    if linha.empty:
        return pd.Series(["Desconhecido", "Desconhecida"])

    return linha.iloc[0][["Estado", "Regiao"]]

In [ ]:
# Usando o nome correto da coluna: 'CEP_3_digitos'
df_cadastro = df_cadastro.withColumn('regiao',
    when((col('CEP_3_digitos') >= 10) & (col('CEP_3_digitos') <= 199), 'SP - São Paulo')
    .when((col('CEP_3_digitos') >= 200) & (col('CEP_3_digitos') <= 289), 'RJ - Rio de Janeiro')
    .when((col('CEP_3_digitos') >= 290) & (col('CEP_3_digitos') <= 299), 'ES - Espírito Santo')
    .when((col('CEP_3_digitos') >= 300) & (col('CEP_3_digitos') <= 399), 'MG - Minas Gerais')
    .when((col('CEP_3_digitos') >= 400) & (col('CEP_3_digitos') <= 499), 'PR - Paraná')
    .when((col('CEP_3_digitos') >= 500) & (col('CEP_3_digitos') <= 599), 'SC - Santa Catarina')
    .when((col('CEP_3_digitos') >= 600) & (col('CEP_3_digitos') <= 699), 'RS - Rio Grande do Sul')
    .when((col('CEP_3_digitos') >= 700) & (col('CEP_3_digitos') <= 727), 'DF - Distrito Federal')
    .when((col('CEP_3_digitos') >= 728) & (col('CEP_3_digitos') <= 769), 'GO - Goiás')
    .when((col('CEP_3_digitos') >= 770) & (col('CEP_3_digitos') <= 779), 'TO - Tocantins')
    .when((col('CEP_3_digitos') >= 780) & (col('CEP_3_digitos') <= 799), 'MT/RO - Mato Grosso/Rondônia')
    .when((col('CEP_3_digitos') >= 800) & (col('CEP_3_digitos') <= 879), 'MS - Mato Grosso do Sul')
    .when((col('CEP_3_digitos') >= 880) & (col('CEP_3_digitos') <= 999), 'Norte - Região Norte')
    .otherwise('CEP Inválido')
)

In [ ]:
df_cadastro.registerTempTable('df_cadastro')

In [ ]:
%sql
SELECT regiao,
       COUNT(*) AS QTD
FROM df_cadastro
WHERE FPD IS NOT NULL AND flag_mig2 = 'PRE'
GROUP BY regiao
ORDER BY regiao

In [ ]:
df_cadastro_pd = df_cadastro.toPandas()

In [ ]:
# Converter FPD para numérico
df_cadastro_pd['FPD'] = pd.to_numeric(df_cadastro_pd['FPD'], errors='coerce')

# Remover nulos
df_cadastro_pd = df_cadastro_pd.dropna(subset=['FPD'])

# Fazer agrupamento
fpd_cep = (
    df_cadastro_pd.groupby('CEP_3_digitos')
      .agg(volume=('FPD', 'count'),
           taxa_fpd=('FPD', 'mean'))
      .reset_index()
)

### Visualização

In [ ]:
# Filtrar CEPs relevantes (volume mínimo)
fpd_cep_filtro = fpd_cep[fpd_cep['volume'] >= 500]

plt.figure(figsize=(8, 5))
plt.scatter(fpd_cep_filtro['volume'], fpd_cep_filtro['taxa_fpd'], color='#b51f19')
plt.xlabel('Volume de Clientes', fontsize=14)
plt.ylabel('Taxa de FPD', fontsize=14)
plt.title('Volume de clientes em cada região por Taxa de FPD', fontsize=16)

# Aumentar fonte dos ticks
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Remover bordas superior e direita
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
display(plt.show())